<a href="https://colab.research.google.com/github/kaveesha82/DS_Project/blob/component-1/arrival_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_excel('/content/drive/MyDrive/DSGP/monthlyarrivals.xlsx')

In [4]:
#Clean column names
df.columns = df.columns.str.strip()

#Remove commas inside numbers
df = df.replace(",", "", regex=True)


df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056 entries, 0 to 1055
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Year         1056 non-null   int64  
 1   Month        1056 non-null   object 
 2   Nationality  1056 non-null   object 
 3   Arrivals     1056 non-null   int64  
 4   Avg_Stay     1056 non-null   float64
 5   Revenue_USD  1056 non-null   float64
dtypes: float64(2), int64(2), object(2)
memory usage: 49.6+ KB


In [6]:
df.shape


(1056, 6)

In [7]:
df.columns

Index(['Year', 'Month', 'Nationality', 'Arrivals', 'Avg_Stay', 'Revenue_USD'], dtype='object')

In [8]:
df.dtypes

,0
Year,int64
Month,object
Nationality,object
Arrivals,int64
Avg_Stay,float64
Revenue_USD,float64


In [9]:
df.isnull().sum()

,0
Year,0
Month,0
Nationality,0
Arrivals,0
Avg_Stay,0
Revenue_USD,0


In [12]:
#Encode months
month_map = {
    "January":1, "February":2, "March":3, "April":4,
    "May":5, "June":6, "July":7, "August":8,
    "September":9, "October":10, "November":11, "December":12
}

df["Month_Num"] = df["Month"].map(month_map)

In [13]:
#Label encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["Nationality_Enc"] = le.fit_transform(df["Nationality"])

In [14]:
df.head()

,Year,Month,Nationality,Arrivals,Avg_Stay,Revenue_USD,Month_Num,Nationality_Enc
0,2025,January,India,45496,6.8,58310242.0,1,6
1,2025,January,Russian Federation,37914,12.5,89335013.0,1,18
2,2025,January,United Kingdom,20220,13.2,50311716.0,1,24
3,2025,January,Germany,17693,14.2,47358358.0,1,5
4,2025,January,China,15165,8.5,24300108.0,1,3


In [15]:
#Time index-creates a continuous numeric timeline
df["Time_Index"] = (df["Year"] - df["Year"].min()) * 12 + df["Month_Num"]

In [16]:
#Lag feature-arrival from prev months for same nationality
df = df.sort_values(["Nationality", "Year", "Month_Num"])

df["Lag_1"] = df.groupby("Nationality")["Arrivals"].shift(1)

In [17]:
#Rolling average-avg over prev 3 months
df["Rolling_3"] = (
    df.groupby("Nationality")["Arrivals"]
    .transform(lambda x: x.shift(1).rolling(window=3).mean())
)

In [18]:
#Handle NaNs created by lags
df_model = df.dropna().reset_index(drop=True)